# LLM advertising spans

Reads each raw transcript CSV with a local instruction-tuned LLM and returns advertising spans as half-open, 0-based transcript line ranges: `[start, end)`. The model sees lines formatted as `[index] text`, so the labels can be mapped back to the original `start` and `end` timestamps without changing the transcript.

Each episode gets a JSON file under `out/llm_ad_spans/` with the same relative path as the input and a `.json` suffix. Long transcripts are processed in overlapping chunks; duplicate and adjacent spans are merged.

This notebook uses the repository's local `vllm` setup. Run it from the repo root and point `INPUT_ROOT` at the raw transcript CSV directory.

In [1]:
import csv
import json
import re
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from vllm import LLM, SamplingParams


MODEL_NAME = "openai/gpt-oss-120b"
# INPUT_ROOT = Path("out/resegmented")
INPUT_ROOT = Path("out/whisper_segmented")
OUTPUT_ROOT = Path("out/llm_ad_spans")
FIG_ROOT = Path('out/llm_ad_plots')
N_EPISODES = None  # set to an int to process only the N largest transcripts
EPISODE_FILES = None  # or a list of Paths under INPUT_ROOT

MAX_MODEL_LEN = 32768
MAX_NEW_TOKENS = 2048
MAX_LINES_PER_REQUEST = 300 #1200
CHUNK_OVERLAP = 24

AD_SPAN_SCHEMA = {
    "type": "object",
    "properties": {
        "spans": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "start": {"type": "integer", "minimum": 0},
                    "end": {"type": "integer", "minimum": 1},
                    "reason": {"type": "string"},
                },
                "required": ["start", "end"],
                "additionalProperties": False,
            },
        },
    },
    "required": ["spans"],
    "additionalProperties": False,
}


/blue/bonniejdorr/ethanlmines/repos/cluster-podcast-transcription/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import copy
import json
from typing import Optional

from vllm.sampling_params import StructuredOutputsParams
from xgrammar import get_model_structural_tag
from xgrammar.structural_tag import GrammarFormat, RegexFormat, JSONSchemaFormat

def make_structured_outputs_params(model_name: str,
                                    *,
                                   grammar: Optional[str] = None,
                                   regex: Optional[str] = None,
                                   json_schema: Optional[str | dict] = None,
                                   reasoning=True
                                   ) -> StructuredOutputsParams:
    if sum(bool(x) for x in [grammar, regex, json_schema]) != 1:
        raise ValueError("Must specifiy exactly one of {grammar, regex}")
    if "llama" in model_name or 'gemma' in model_name:
        return StructuredOutputsParams(grammar=grammar, regex=regex, json=json_schema)
    elif 'gpt-oss' in model_name:
        # Have to make an expanded grammar that supports the Harmony format
        structural_tag = get_model_structural_tag(
            "harmony",
            tools=[],
            tool_choice="none",
            reasoning=reasoning,
        )
        structural_tag = copy.deepcopy(structural_tag)

        if grammar:
            fmt_object = GrammarFormat(grammar=grammar)
        elif regex:
            fmt_object = RegexFormat(pattern=regex)
        else:
            fmt_object = JSONSchemaFormat(json_schema=json_schema)

        for tag in structural_tag.format.tags:
            # Only care about constraining the final answer
            if tag.begin == '<|channel|>final<|message|>':
                tag.content = fmt_object
        return StructuredOutputsParams(structural_tag=json.dumps(structural_tag.model_dump()))
    else:
        raise ValueError(f"Unsupported model {model_name}")
AD_SPAN_OUTPUTS = make_structured_outputs_params(MODEL_NAME, json_schema=AD_SPAN_SCHEMA, reasoning=False)

In [3]:
AD_SPAN_OUTPUTS

StructuredOutputsParams(json=None, regex=None, choice=None, grammar=None, json_object=None, disable_any_whitespace=False, disable_additional_properties=False, whitespace_pattern=None, structural_tag='{"type": "structural_tag", "format": {"type": "tags_with_separator", "tags": [{"type": "tag", "begin": "<|channel|>final<|message|>", "content": {"type": "json_schema", "json_schema": {"type": "object", "properties": {"spans": {"type": "array", "items": {"type": "object", "properties": {"start": {"type": "integer", "minimum": 0}, "end": {"type": "integer", "minimum": 1}, "reason": {"type": "string"}}, "required": ["start", "end"], "additionalProperties": false}}}, "required": ["spans"], "additionalProperties": false}, "style": "json", "any_order": false}, "end": ["<|end|>", "<|return|>"]}], "separator": "<|start|>assistant", "at_least_one": false, "stop_after_first": false}}', _backend=None, _backend_was_auto=False)

In [4]:
llm = LLM(model=MODEL_NAME, max_model_len=MAX_MODEL_LEN, max_num_seqs=1, gpu_memory_utilization=0.95)

INFO 10-06 16:41:20 [api_utils.py:272] non-default args: {'max_model_len': 32768, 'gpu_memory_utilization': 0.95, 'max_num_seqs': 1, 'disable_log_stats': True, 'model': 'openai/gpt-oss-120b'}
INFO 10-06 16:41:20 [model.py:672] Resolved architecture: GptOssForCausalLM


Parse safetensors files: 100%|██████████| 15/15 [00:00<00:00, 16.33it/s]

INFO 10-06 16:41:21 [model.py:1965] Using max model len 32768


INFO 10-06 16:41:23 [scheduler.py:242] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 10-06 16:41:23 [config.py:410] Overriding max cuda graph capture size to 1024 for performance.
INFO 10-06 16:41:23 [kernel.py:308] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(EngineCore pid=326677) INFO 10-06 16:41:28 [core.py:122] Initializing a V1 LLM engine (v0.28.0) with config: model='openai/gpt-oss-120b', speculative_config=None, tokenizer='openai/gpt-oss-120b', skip_tokenizer_init=False, tokenizer_mode=auto, revision=main, tokenizer_revision=main, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=gpt_oss_mxfp4, quantization_config=None, enforce_eager=False, enable_return_

Loading safetensors checkpoint shards:   0% Completed | 0/15 [00:00<?, ?it/s]


...

(EngineCore pid=326677) 2026-10-06 16:44:58,632 - INFO - autotuner.py:2397 - flashinfer.jit: [Autotuner]: Loaded 22 configs from /blue/bonniejdorr/ethanlmines/vllm/flashinfer_autotune_cache/0.6.16.post3/100a/98d5edc23f5a04933cc1b60ad0f458ec3be9428e64006771a4a367bd52ff774f/autotune_configs.json
(EngineCore pid=326677) 2026-10-06 16:44:58,641 - INFO - autotuner.py:829 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore pid=326677) 2026-10-06 16:44:58,668 - INFO - autotuner.py:1334 - flashinfer.jit: [Autotuner]: Config cache hit for flashinfer::trtllm_fp4_block_scale_moe (runner=MoERunner, source=config file)
(EngineCore pid=326677) 2026-10-06 16:44:58,775 - INFO - autotuner.py:852 - flashinfer.jit: [Autotuner]: Autotuning process ends
(EngineCore pid=326677) 2026-10-06 16:44:58,848 - INFO - autotuner.py:2269 - flashinfer.jit: [Autotuner]: Saved 22 configs to /blue/bonniejdorr/ethanlmines/vllm/flashinfer_autotune_cache/0.6.16.post3/100a/98d5edc23f5a04933cc1b60ad0f458e

(EngineCore pid=326677) INFO 10-06 16:44:58 [kernel_warmup.py:195] Warming up FlashInfer attention.
(EngineCore pid=326677) INFO 10-06 16:44:58 [gpu_model_runner.py:7002] Rank 0: Torch profiler disabled for CUDA graph capture


Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 83/83 [00:04<00:00, 19.67it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 1/1 [00:00<00:00, 23.98it/s]


(EngineCore pid=326677) INFO 10-06 16:45:03 [gpu_model_runner.py:7048] Graph capturing finished in 5 secs, took 0.97 GiB
(EngineCore pid=326677) INFO 10-06 16:45:03 [gpu_worker.py:741] CUDA graph pool memory: 0.97 GiB (actual), 1.35 GiB (estimated), difference: 0.38 GiB (38.8%).
(EngineCore pid=326677) INFO 10-06 16:45:03 [gpu_worker.py:804] Free memory on device (177.74/178.35 GiB) on startup. Desired GPU memory utilization is (0.95, 169.43 GiB). Actual usage is 67.15 GiB for consumed memory (weights + non-torch), 2.62 GiB for peak activation, and 0.97 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=105817691239` (98.55 GiB) to fit into requested memory, or `--kv-cache-memory=114736754176` (106.86 GiB) to fully utilize gpu memory. Current kv cache memory in use is 99.67 GiB.
(EngineCore pid=326677) INFO 10-06 16:45:07 [jit_monitor.py:85] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
(EngineCore pid

(EngineCore pid=326677) WARNING 10-06 16:45:59 [jit_monitor.py:141] Triton kernel JIT compilation during inference: apply_token_bitmask_inplace_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


## Prompt and response parsing

The model is constrained to return JSON matching `AD_SPAN_SCHEMA`, in the form `{"spans": [{"start": 12, "end": 18, "reason": "..."}]}`. `start` is inclusive and `end` is exclusive, so this example labels lines 12 through 17. Parsing still tolerates surrounding model text, while validation rejects out-of-range or malformed spans.

In [5]:
SYSTEM_PROMPT = """You identify advertising in podcast transcripts.

Reasoning: low

Return one JSON object matching the provided schema, with a `spans` array. Each span must contain integer `start` and `end` fields and may contain a short `reason` field.
Spans use half-open intervals: `start` is inclusive and `end` is exclusive. Indices refer to the 0-based line numbers supplied in the transcript.
"""

USER_PROMPT = """Read the transcript below and identify every contiguous range of transcript lines that is advertising. Include sponsor reads, product or service promotions, affiliate messages, fundraising or membership appeals, and promotions for the podcast or its network. Do not label ordinary discussion of a product, news, or a guest's work unless it is promotional.

Return JSON matching the schema. Use half-open spans: `start` is the first advertising line and `end` is one past the last advertising line. For example, `{"start": 0, "end": 3}` labels lines 0, 1, and 2.

Use the supplied line indices exactly. If there is no advertising, return {"spans": []}. Do not include any text outside the JSON object.

TRANSCRIPT:
"""


def episode_paths(root, files, n):
    root = Path(root).resolve()
    if not root.is_dir():
        raise FileNotFoundError(f"Transcript directory does not exist: {root}")
    if files:
        paths = []
        for path in files:
            resolved = Path(path).resolve()
            if not resolved.is_relative_to(root):
                raise ValueError(f"input file is outside INPUT_ROOT: {path}")
            paths.append(resolved)
    else:
        paths = list(root.glob("**/*.csv"))
    paths = sorted(paths, key=lambda path: path.stat().st_size, reverse=True)
    return paths if n is None else paths[:n]


def read_transcript(path):
    with open(path, newline="", encoding="utf-8") as source:
        rows = list(csv.DictReader(source))
    if rows:
        missing = {"start", "end", "text"} - set(rows[0])
        if missing:
            raise ValueError(f"{path} is missing columns: {sorted(missing)}")
    return rows


def format_transcript(rows, offset=0):
    return "\n".join(f"[{offset + index}] {row['text'].strip()}" for index, row in enumerate(rows))


def extract_json_object(text):
    decoder = json.JSONDecoder()
    for match in re.finditer(r"\{", text):
        try:
            value, _ = decoder.raw_decode(text[match.start():])
        except json.JSONDecodeError:
            continue
        if isinstance(value, dict) and "spans" in value:
            return value
        print(f"Warning: LLM response did not contain a spans JSON object: {text[:500]!r}")
        return {"spans": []}


def validate_spans(payload, line_count, offset=0):
    if not isinstance(payload, dict) or not isinstance(payload.get("spans"), list):
        print("Warning: LLM response must contain a spans array")
        return []
    validated = []
    lower = offset
    upper = offset + line_count
    for span in payload["spans"]:
        if not isinstance(span, dict) or not isinstance(span.get("start"), int) or not isinstance(span.get("end"), int):
            print(f"Warning: Malformed span: {span!r}")
            continue
        start, end = span["start"], span["end"]
        if not lower <= start < end <= upper:
            print(f"Warning: Span {span!r} is outside half-open line range [{lower}, {upper})")
            continue
        validated.append({"start": start, "end": end, "reason": str(span.get("reason", ""))})
    return validated


def merge_spans(spans):
    merged = []
    for span in sorted(spans, key=lambda item: (item["start"], item["end"])):
        if merged and span["start"] <= merged[-1]["end"]:
            merged[-1]["end"] = max(merged[-1]["end"], span["end"])
            if span.get("reason") and span["reason"] not in merged[-1].get("reason", ""):
                merged[-1]["reason"] = f"{merged[-1].get('reason', '')}; {span['reason']}".strip("; ")
        else:
            merged.append(dict(span))
    return merged

## Label episodes

In [6]:
def label_transcripts(input_path):
    input_path = Path(input_path)
    rows = read_transcript(input_path)
    all_spans = []
    if rows:
        step = max(1, MAX_LINES_PER_REQUEST - CHUNK_OVERLAP)
        chunks = [(offset, rows[offset:offset + MAX_LINES_PER_REQUEST]) for offset in range(0, len(rows), step)]
        conversations = [
            [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": USER_PROMPT + format_transcript(chunk, offset)},
            ]
            for offset, chunk in chunks
        ]
        sampling_params = SamplingParams(
            temperature=0.0,
            max_tokens=MAX_NEW_TOKENS,
            structured_outputs=AD_SPAN_OUTPUTS,
        )
        outputs = llm.chat(conversations, sampling_params=sampling_params, use_tqdm=True)
        for (offset, chunk), output in zip(chunks, outputs):
            payload = extract_json_object(output.outputs[0].text)
            all_spans.extend(validate_spans(payload, len(chunk), offset))
    return {
        "source": str(input_path),
        "line_count": len(rows),
        "spans": merge_spans(all_spans),
    }


def extract_ad_text(transcript_path, json_output):
    rows = read_transcript(transcript_path)
    spans = json_output.get("spans", [])
    ad_text = []
    for span in spans:
        start, end = span["start"], span["end"]
        if not 0 <= start < end <= len(rows):
            raise ValueError(f"Span {span!r} is outside transcript line range [0, {len(rows)})")
        ad_text.append(" ".join(row["text"].strip() for row in rows[start:end]))
    return ad_text

## Plot ad spans

`plot_ad_spans` accepts a label JSON path, the parsed label dictionary, or a list of span dictionaries. Spans use half-open `[start, end)` indices, and the plot shades from the start of line `start` through the end of line `end - 1`. Timestamp parsing follows the same `clock`, seconds, centiseconds, and milliseconds conventions as the classifier notebook.

In [17]:
def _clock_to_seconds(text):
    total = 0.0
    for part in str(text).strip().split(":"):
        total = total * 60 + float(part)
    return total


def timestamps_to_seconds(values, timestamp_unit="auto"):
    texts = [str(value).strip() for value in values]
    if timestamp_unit == "auto" and any(":" in text for text in texts):
        return [_clock_to_seconds(text) for text in texts], "clock"
    if timestamp_unit == "clock":
        return [_clock_to_seconds(text) for text in texts], "clock"
    numbers = [float(text) for text in texts]
    divisors = {"s": 1.0, "cs": 100.0, "ms": 1000.0}
    if timestamp_unit in divisors:
        return [number / divisors[timestamp_unit] for number in numbers], timestamp_unit
    if timestamp_unit != "auto":
        raise ValueError("timestamp_unit must be 'auto', 'clock', 's', 'cs', or 'ms'")
    gaps = [b - a for a, b in zip(numbers, numbers[1:]) if b > a]
    median_gap = float(np.median(gaps)) if gaps else 0.0
    if median_gap >= 500:
        return [number / 1000.0 for number in numbers], "ms"
    if median_gap >= 50:
        return [number / 100.0 for number in numbers], "cs"
    return numbers, "s"


def load_ad_spans(labels):
    if isinstance(labels, (str, Path)):
        with open(labels, encoding="utf-8") as source:
            labels = json.load(source)
    if isinstance(labels, dict):
        labels = labels.get("spans", [])
    labels = [dict(start=span["start"], end=span["end"], reason=str(span.get("reason", ""))) for span in labels]
    return merge_spans(labels)


def plot_ad_spans(transcript_csv, labels, timestamp_unit="auto"):
    rows = read_transcript(transcript_csv)
    if not rows:
        raise ValueError(f"{transcript_csv} has no transcript lines")
    seconds, unit = timestamps_to_seconds([row["start"] for row in rows], timestamp_unit)
    ends, _ = timestamps_to_seconds([row["end"] for row in rows], timestamp_unit)
    minutes = np.asarray(seconds) / 60.0
    spans = load_ad_spans(labels)
    hover_text = [row["text"].strip().replace("\n", " ")[:500] for row in rows]
    figure = go.Figure()
    for span in spans:
        start, end = span["start"], span["end"]
        if not 0 <= start < end <= len(rows):
            raise ValueError(f"Span {span!r} is outside transcript line range [0, {len(rows)})")
        reason = span.get("reason", "")
        hover_text = f"ad span [{start}, {end})"
        if reason:
            hover_text += f"<br>reason: {reason}"
        figure.add_trace(go.Scatter(x=[minutes[start], minutes[start], ends[end - 1] / 60.0, ends[end - 1] / 60.0, minutes[start]], y=[-1, 1, 1, -1, -1], fill="toself", fillcolor="#e76f51", opacity=0.28, line=dict(width=0), mode="lines", hoveron="fills", text=hover_text, hovertemplate="%{text}<extra></extra>", showlegend=False))
    figure.add_trace(go.Scatter(x=minutes, y=np.zeros(len(rows)), mode="markers", marker=dict(size=3, color="#2f4858"), name="Transcript lines", text=[row["text"].strip().replace("\n", " ")[:500] for row in rows], customdata=np.arange(len(rows)), hovertemplate="line %{customdata}<br>%{x:.2f} min<br>%{text}<extra></extra>"))
    figure.update_layout(title=f"{Path(transcript_csv).name}: advertising spans", xaxis_title=f"Minutes into episode ({unit})", yaxis=dict(visible=False, range=[-1, 1]), height=360, template="plotly_white", showlegend=False, hovermode="closest")
    print(f"{transcript_csv}: {len(rows)} lines, {len(spans)} advertising spans, timestamps read as {unit}")
    return figure

In [8]:
# Example after labeling: choose a transcript and its matching JSON label file.
transcript_paths = sorted(Path(INPUT_ROOT).glob("**/*.csv"))

In [9]:
# transcript_path = transcript_paths[0]
transcript_path = Path(INPUT_ROOT) / "5929292/5929292_00010.mp3.csv"
json_output = label_transcripts(transcript_path)

Rendering conversations:   0%|          | 0/14 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 14/14 [01:13<00:00,  5.28s/it, est. speed input: 775.80 toks/s, output: 361.04 toks/s]


In [10]:
json_output

{'source': 'out/whisper_segmented/5929292/5929292_00010.mp3.csv',
 'line_count': 3844,
 'spans': [{'start': 0,
   'end': 7,
   'reason': 'Product ad for Burley Aero boot with website call‑to‑action'},
  {'start': 129,
   'end': 136,
   'reason': 'Repeated ad for La Crosse Footwear with website call‑to‑action'},
  {'start': 137,
   'end': 149,
   'reason': 'Ad for Teriyaki Madness with website call‑to‑action'},
  {'start': 188,
   'end': 234,
   'reason': 'Promotion of premium subscription/gift to listeners'},
  {'start': 235,
   'end': 270,
   'reason': 'Fundraising/appeal for Turning Point USA (TPUSA) support'},
  {'start': 335,
   'end': 347,
   'reason': 'Sponsor read for Burley Aero rubber boot and MyFICO'},
  {'start': 394,
   'end': 416,
   'reason': 'Repeated sponsor read for Burley Aero boot and MyFICO'},
  {'start': 422,
   'end': 440,
   'reason': 'Promotion of Rumble platform and upcoming show'},
  {'start': 475,
   'end': 480,
   'reason': 'Thank you to raid and mention of 

In [11]:
import os
csv_relpath = transcript_path.relative_to(INPUT_ROOT)
label_path = Path(OUTPUT_ROOT) / csv_relpath.with_suffix(".json")
os.makedirs(os.path.dirname(label_path), exist_ok=True)
with open(label_path, 'w') as w:
    json.dump(json_output, w, indent=2)


In [16]:
fig = plot_ad_spans(transcript_path, label_path)
fig

out/whisper_segmented/5929292/5929292_00010.mp3.csv: 3844 lines, 34 advertising spans, timestamps read as ms


In [ ]:
plot_path = Path(FIG_ROOT) / csv_relpath.with_suffix('.html')
os.makedirs(os.path.dirname(plot_path), exist_ok=True)
fig.write_html(plot_path)